<a href="https://colab.research.google.com/github/EmePin/Analisis-de-datos/blob/main/Fake_News_2_1_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import kagglehub
import os
!pip install tensorflow

from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

import gradio as gr
import joblib


In [ ]:
# Descargar dataset
path = kagglehub.dataset_download("subho117/fake-news-detection-using-machine-learning")

df = pd.read_csv(os.path.join(path, "News.csv"))

# Usaremos título (rápido y efectivo)
df = df[['title', 'class']].dropna()

X = df['title'].astype(str).values
y = df['class'].astype(int).values


100%|██████████| 41.0M/41.0M [00:03<00:00, 12.5MB/s]

Extracting files...


In [ ]:
print(df)

                                                   title  class
0       Donald Trump Sends Out Embarrassing New Year’...      0
1       Drunk Bragging Trump Staffer Started Russian ...      0
2       Sheriff David Clarke Becomes An Internet Joke...      0
3       Trump Is So Obsessed He Even Has Obama’s Name...      0
4       Pope Francis Just Called Out Donald Trump Dur...      0
...                                                  ...    ...
44914  'Fully committed' NATO backs new U.S. approach...      1
44915  LexisNexis withdrew two products from Chinese ...      1
44916  Minsk cultural hub becomes haven from authorities      1
44917  Vatican upbeat on possibility of Pope Francis ...      1
44918  Indonesia to buy $1.14 billion worth of Russia...      1

[44919 rows x 2 columns]


In [ ]:
print(df)

                                                   title  class
0       Donald Trump Sends Out Embarrassing New Year’...      0
1       Drunk Bragging Trump Staffer Started Russian ...      0
2       Sheriff David Clarke Becomes An Internet Joke...      0
3       Trump Is So Obsessed He Even Has Obama’s Name...      0
4       Pope Francis Just Called Out Donald Trump Dur...      0
...                                                  ...    ...
44914  'Fully committed' NATO backs new U.S. approach...      1
44915  LexisNexis withdrew two products from Chinese ...      1
44916  Minsk cultural hub becomes haven from authorities      1
44917  Vatican upbeat on possibility of Pope Francis ...      1
44918  Indonesia to buy $1.14 billion worth of Russia...      1

[44919 rows x 2 columns]


In [ ]:
print(df.head())

                                               title  class
0   Donald Trump Sends Out Embarrassing New Year’...      0
1   Drunk Bragging Trump Staffer Started Russian ...      0
2   Sheriff David Clarke Becomes An Internet Joke...      0
3   Trump Is So Obsessed He Even Has Obama’s Name...      0
4   Pope Francis Just Called Out Donald Trump Dur...      0


In [ ]:
VOCAB_SIZE = 5000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=VOCAB_SIZE)# convierte palabras → números
tokenizer.fit_on_texts(X) # Diccionario

X_seq = tokenizer.texts_to_sequences(X)
# Convierte cada noticia en una lista de números
# Ejemplo: "economy grows fast" → [45, 102, 78]
X_pad = pad_sequences(X_seq, maxlen=MAX_LEN)
# Ajusta todas las noticias al mismo tamaño
# Si son cortas → agrega ceros (padding)
# Si son largas → las recorta


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad, y, test_size=0.2, random_state=42, stratify=y
)


In [ ]:
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=128, input_length=MAX_LEN),
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)


Epoch 1/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 237s 505ms/step - accuracy: 0.9378 - loss: 0.1561 - val_accuracy: 0.9673 - val_loss: 0.0869
Epoch 2/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 245s 470ms/step - accuracy: 0.9781 - loss: 0.0596 - val_accuracy: 0.9677 - val_loss: 0.0847
Epoch 3/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 215s 478ms/step - accuracy: 0.9877 - loss: 0.0356 - val_accuracy: 0.9695 - val_loss: 0.1000
Epoch 4/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 212s 471ms/step - accuracy: 0.9926 - loss: 0.0207 - val_accuracy: 0.9669 - val_loss: 0.1129
Epoch 5/5
450/450 ━━━━━━━━━━━━━━━━━━━━ 209s 465ms/step - accuracy: 0.9953 - loss: 0.0134 - val_accuracy: 0.9701 - val_loss: 0.1146


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Accuracy:", accuracy)


281/281 ━━━━━━━━━━━━━━━━━━━━ 26s 94ms/step - accuracy: 0.9686 - loss: 0.1166
Accuracy: 0.9686108827590942


In [ ]:
model.save("lstm_fake_news.keras")


joblib.dump(tokenizer, "tokenizer_lstm.pkl")


['tokenizer_lstm.pkl']

In [ ]:
def predict_news_lstm(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=MAX_LEN)

    pred = model.predict(padded)[0][0]

    if pred > 0.5:
        label = "REAL"
        confidence = pred
    else:
        label = "FALSA"
        confidence = 1 - pred

    return f"{label} ({confidence*100:.1f}% de confianza)"

In [ ]:
app = gr.Interface(
    fn=predict_news_lstm,

    inputs=gr.Textbox(
        label="Escribe una noticia en inglés",
        placeholder="Ej: Vaccines contain microchips..."
    ),

    outputs=gr.Text(label="Resultado"),

    title="📰 Detector de Fake News",
    description="Escribe una noticia en inglés y la IA te dirá si es REAL o FALSA"
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6426c259271fb48c0f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
print(df)
print(df.head())
print("Hola mundo")
print("Aimée")
nombre = "Aimée"
print(nombre)
suma = 2+2
print(suma)
sumaConcatenada ="2"+"2"
print(sumaConcatenada)

numero=5
def cuadrado(numero):
    return numero*numero

print(cuadrado(numero))

numero1 = 3
numero2 = 4
def suma(numero1, numero2):
  return numero1 + numero2

print(suma(numero1, numero2))




                                                   title  class
0       Donald Trump Sends Out Embarrassing New Year’...      0
1       Drunk Bragging Trump Staffer Started Russian ...      0
2       Sheriff David Clarke Becomes An Internet Joke...      0
3       Trump Is So Obsessed He Even Has Obama’s Name...      0
4       Pope Francis Just Called Out Donald Trump Dur...      0
...                                                  ...    ...
44914  'Fully committed' NATO backs new U.S. approach...      1
44915  LexisNexis withdrew two products from Chinese ...      1
44916  Minsk cultural hub becomes haven from authorities      1
44917  Vatican upbeat on possibility of Pope Francis ...      1
44918  Indonesia to buy $1.14 billion worth of Russia...      1

[44919 rows x 2 columns]
                                               title  class
0   Donald Trump Sends Out Embarrassing New Year’...      0
1   Drunk Bragging Trump Staffer Started Russian ...      0
2   Sheriff David Clarke B

In [ ]:
print(df)
print(df.head())
print("Hola mundo")
print("Aimée")

nombre_Apellido = "Aimée Pineda"
print(nombre_Apellido)

suma = 2+2
print(suma)

sumaConcatenada = "2" +"2"+"Camila"
print(sumaConcatenada)

numero1 = 3
numero2 = 4
def suma(numero1,numero2):
  return numero1 + numero2

resultado = suma(numero1,numero2)
print(resultado)

def potencia(numero):
  return numero*numero*numero*numero*numero






                                                   title  class
0       Donald Trump Sends Out Embarrassing New Year’...      0
1       Drunk Bragging Trump Staffer Started Russian ...      0
2       Sheriff David Clarke Becomes An Internet Joke...      0
3       Trump Is So Obsessed He Even Has Obama’s Name...      0
4       Pope Francis Just Called Out Donald Trump Dur...      0
...                                                  ...    ...
44914  'Fully committed' NATO backs new U.S. approach...      1
44915  LexisNexis withdrew two products from Chinese ...      1
44916  Minsk cultural hub becomes haven from authorities      1
44917  Vatican upbeat on possibility of Pope Francis ...      1
44918  Indonesia to buy $1.14 billion worth of Russia...      1

[44919 rows x 2 columns]
                                               title  class
0   Donald Trump Sends Out Embarrassing New Year’...      0
1   Drunk Bragging Trump Staffer Started Russian ...      0
2   Sheriff David Clarke B

In [ ]:
print(df)
print(df.head())
print("Aimée")

apellido = "Pineda"
print(apellido)

operacionSuma = 2+2
print(operacionSuma)

operacionMultiplicacion = 2*8
print(operacionMultiplicacion)

sumaConcatenada = "2"+"2"+"Camila"
print(sumaConcatenada)

numero1 = 3
numero2 = 4

def suma(numero1,numero2):
  return numero1 + numero2

def cuadrado(numero1):
  return numero1*numero1



                                                   title  class
0       Donald Trump Sends Out Embarrassing New Year’...      0
1       Drunk Bragging Trump Staffer Started Russian ...      0
2       Sheriff David Clarke Becomes An Internet Joke...      0
3       Trump Is So Obsessed He Even Has Obama’s Name...      0
4       Pope Francis Just Called Out Donald Trump Dur...      0
...                                                  ...    ...
44914  'Fully committed' NATO backs new U.S. approach...      1
44915  LexisNexis withdrew two products from Chinese ...      1
44916  Minsk cultural hub becomes haven from authorities      1
44917  Vatican upbeat on possibility of Pope Francis ...      1
44918  Indonesia to buy $1.14 billion worth of Russia...      1

[44919 rows x 2 columns]
                                               title  class
0   Donald Trump Sends Out Embarrassing New Year’...      0
1   Drunk Bragging Trump Staffer Started Russian ...      0
2   Sheriff David Clarke B

In [ ]:
print(df)
print(df.head())
print("Aimée")
print("Hola mundo")

numero1 = 3
apellido = "Pineda"

suma = 2+2
multiplicacion= 2*2
potencia = 2**8

print(suma)
print(multiplicacion)
print(potencia)

concatenacion = "6" + "7" + "Camila"
print(concatenacion)

numero2 = 3
numero3 = 4
def suma(numero2, numero3):
    return numero2 + numero3

def potencia(numero1):
    return numero1**3










                                                   title  class
0       Donald Trump Sends Out Embarrassing New Year’...      0
1       Drunk Bragging Trump Staffer Started Russian ...      0
2       Sheriff David Clarke Becomes An Internet Joke...      0
3       Trump Is So Obsessed He Even Has Obama’s Name...      0
4       Pope Francis Just Called Out Donald Trump Dur...      0
...                                                  ...    ...
44914  'Fully committed' NATO backs new U.S. approach...      1
44915  LexisNexis withdrew two products from Chinese ...      1
44916  Minsk cultural hub becomes haven from authorities      1
44917  Vatican upbeat on possibility of Pope Francis ...      1
44918  Indonesia to buy $1.14 billion worth of Russia...      1

[44919 rows x 2 columns]
                                               title  class
0   Donald Trump Sends Out Embarrassing New Year’...      0
1   Drunk Bragging Trump Staffer Started Russian ...      0
2   Sheriff David Clarke B

In [ ]:
print(df)
print(df.head())
print("Aimée")
print("hola mundo")

apellido = "Pineda"
print(apellido)

suma = 4 + 2
print(suma)

multiplicacion = 4 * 2
print(multiplicacion)

potencia = 4**2
print(potencia)

concatenacion = "6" + "7" + "Pedro"
print(concatenacion)

numero1 = 3
numero2 = 4

def suma(numero1,numero2):
    suma = numero1+numero2
    return suma

def potencia(numero1):
  return numero1**2





                                                   title  class
0       Donald Trump Sends Out Embarrassing New Year’...      0
1       Drunk Bragging Trump Staffer Started Russian ...      0
2       Sheriff David Clarke Becomes An Internet Joke...      0
3       Trump Is So Obsessed He Even Has Obama’s Name...      0
4       Pope Francis Just Called Out Donald Trump Dur...      0
...                                                  ...    ...
44914  'Fully committed' NATO backs new U.S. approach...      1
44915  LexisNexis withdrew two products from Chinese ...      1
44916  Minsk cultural hub becomes haven from authorities      1
44917  Vatican upbeat on possibility of Pope Francis ...      1
44918  Indonesia to buy $1.14 billion worth of Russia...      1

[44919 rows x 2 columns]
                                               title  class
0   Donald Trump Sends Out Embarrassing New Year’...      0
1   Drunk Bragging Trump Staffer Started Russian ...      0
2   Sheriff David Clarke B